<a href="https://colab.research.google.com/github/ahaddd-ship-it/natural-language-proccesing/blob/main/NLP5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [24]:
import nltk
from nltk import word_tokenize, bigrams, FreqDist, ConditionalFreqDist
from nltk.corpus import stopwords
from collections import defaultdict
from nltk.util import trigrams
from nltk.corpus import reuters

In [28]:
        # Download necessary NLTK resources
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('punkt_tab') # Added to resolve LookupError: Resource punkt_tab not found
nltk.download('reuters')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package reuters to /root/nltk_data...


True

In [15]:


        # Step 1: Load and preprocess the text corpus
        def preprocess(text):
            # Convert to lowercase
            text = text.lower()
            # Tokenize the text
            tokens = word_tokenize(text)
            # Remove punctuation and stopwords
            stop_words = set(stopwords.words('english'))
            tokens = [token for token in tokens if token.isalnum() and token not in stop_words]
            return tokens
# Sample corpus text
corpus = "The quick brown fox jumps over the lazy dog. The fox is quick and the dog is lazy."

# Preprocess the text using your step 1 function
tokens = preprocess(corpus)

# 1. Generate Bigrams (pairs of consecutive words)
bi_grams = list(bigrams(tokens))

# 2. Calculate Overall Frequency Distribution of Bigrams
bigram_freq = FreqDist(bi_grams)

# 3. Calculate Conditional Frequency Distribution
# (Useful for predicting the next word given a current word)
cfd = ConditionalFreqDist(bi_grams)

# --- Print the Results ---
print("--- Preprocessed Tokens ---")
print(tokens)

print("\n--- Generated Bigrams ---")
print(bi_grams)

print("\n--- Top 3 Most Common Bigrams ---")
print(bigram_freq.most_common(3))

print("\n--- Words that most likely follow 'quick' ---")
# This looks at the conditions where the first word is 'quick'
print(dict(cfd['quick']))

--- Preprocessed Tokens ---
['quick', 'brown', 'fox', 'jumps', 'lazy', 'dog', 'fox', 'quick', 'dog', 'lazy']

--- Generated Bigrams ---
[('quick', 'brown'), ('brown', 'fox'), ('fox', 'jumps'), ('jumps', 'lazy'), ('lazy', 'dog'), ('dog', 'fox'), ('fox', 'quick'), ('quick', 'dog'), ('dog', 'lazy')]

--- Top 3 Most Common Bigrams ---
[(('quick', 'brown'), 1), (('brown', 'fox'), 1), (('fox', 'jumps'), 1)]

--- Words that most likely follow 'quick' ---
{'brown': 1, 'dog': 1}


In [4]:
# Step 3: Generate bigrams from the tokenized text
bigrams_list = list(bigrams(tokens))
print(bigrams_list)


[('quick', 'brown'), ('brown', 'fox'), ('fox', 'jumps'), ('jumps', 'lazy'), ('lazy', 'dog'), ('dog', 'fox'), ('fox', 'quick'), ('quick', 'dog'), ('dog', 'lazy')]


In [5]:
# Step 4: Count bigram frequencies using FreqDist
bigram_freq = FreqDist(bigrams_list)
bigram_freq


FreqDist({('quick', 'brown'): 1, ('brown', 'fox'): 1, ('fox', 'jumps'): 1, ('jumps', 'lazy'): 1, ('lazy', 'dog'): 1, ('dog', 'fox'): 1, ('fox', 'quick'): 1, ('quick', 'dog'): 1, ('dog', 'lazy'): 1})

In [10]:
# Step 3: Generate bigrams from the tokenized text
trigrams_list = list(trigrams(tokens))
print(trigrams_list)


[('quick', 'brown', 'fox'), ('brown', 'fox', 'jumps'), ('fox', 'jumps', 'lazy'), ('jumps', 'lazy', 'dog'), ('lazy', 'dog', 'fox'), ('dog', 'fox', 'quick'), ('fox', 'quick', 'dog'), ('quick', 'dog', 'lazy')]


In [11]:
trigram_freq = FreqDist(trigrams_list)
trigram_freq

FreqDist({('quick', 'brown', 'fox'): 1, ('brown', 'fox', 'jumps'): 1, ('fox', 'jumps', 'lazy'): 1, ('jumps', 'lazy', 'dog'): 1, ('lazy', 'dog', 'fox'): 1, ('dog', 'fox', 'quick'): 1, ('fox', 'quick', 'dog'): 1, ('quick', 'dog', 'lazy'): 1})

In [19]:
cfd = ConditionalFreqDist(bigrams_list)
cfd.keys()

dict_keys(['quick', 'brown', 'fox', 'jumps', 'lazy', 'dog'])

In [20]:
cfd.items()

dict_items([('quick', FreqDist({'brown': 1, 'dog': 1})), ('brown', FreqDist({'fox': 1})), ('fox', FreqDist({'jumps': 1, 'quick': 1})), ('jumps', FreqDist({'lazy': 1})), ('lazy', FreqDist({'dog': 1})), ('dog', FreqDist({'fox': 1, 'lazy': 1}))])

In [21]:
# Step 6: Function to predict the next word based on a given word
def predict_next_word(word):
    word = word.lower()
    if word in cfd:
        next_word = cfd[word].max() # Get the most likely word following the given word
        return next_word
    else:
        return None


In [22]:
# Test the prediction function
start_word = "jumps"
predicted_word = predict_next_word(start_word)
if predicted_word:
    print(f"Next word after '{start_word}': {predicted_word}")
else:
    print(f"No prediction available for the word '{start_word}'.")


Next word after 'jumps': lazy


In [23]:
# Step 7: Generate a sequence of words
def generate_sequence(start_word, length=10):
    sequence = [start_word]
    current_word = start_word
    for _ in range(length):
        next_word = predict_next_word(current_word)
        if next_word:
            sequence.append(next_word)
            current_word = next_word
        else:
            break
    return ' '.join(sequence)

# Generate a sequence starting from "the"
generated_sequence = generate_sequence("fox", length=10)
print(f"Generated sequence: {generated_sequence}")


Generated sequence: fox jumps lazy dog fox jumps lazy dog fox jumps lazy


In [29]:
# List of file IDs in the corpus
file_ids = reuters.fileids()
print(file_ids[0:100])  # Print the first 100 file IDs


['test/14826', 'test/14828', 'test/14829', 'test/14832', 'test/14833', 'test/14839', 'test/14840', 'test/14841', 'test/14842', 'test/14843', 'test/14844', 'test/14849', 'test/14852', 'test/14854', 'test/14858', 'test/14859', 'test/14860', 'test/14861', 'test/14862', 'test/14863', 'test/14865', 'test/14867', 'test/14872', 'test/14873', 'test/14875', 'test/14876', 'test/14877', 'test/14881', 'test/14882', 'test/14885', 'test/14886', 'test/14888', 'test/14890', 'test/14891', 'test/14892', 'test/14899', 'test/14900', 'test/14903', 'test/14904', 'test/14907', 'test/14909', 'test/14911', 'test/14912', 'test/14913', 'test/14918', 'test/14919', 'test/14921', 'test/14922', 'test/14923', 'test/14926', 'test/14928', 'test/14930', 'test/14931', 'test/14932', 'test/14933', 'test/14934', 'test/14941', 'test/14943', 'test/14949', 'test/14951', 'test/14954', 'test/14957', 'test/14958', 'test/14959', 'test/14960', 'test/14962', 'test/14963', 'test/14964', 'test/14965', 'test/14967', 'test/14968', 'test

In [31]:
# Create trigrams
tri_grams = list(trigrams(tokens))
print(tri_grams[:10000])

# Building a Trigram Model

# Build a trigram model
model = defaultdict(lambda: defaultdict(lambda: 0))

[('quick', 'brown', 'fox'), ('brown', 'fox', 'jumps'), ('fox', 'jumps', 'lazy'), ('jumps', 'lazy', 'dog'), ('lazy', 'dog', 'fox'), ('dog', 'fox', 'quick'), ('fox', 'quick', 'dog'), ('quick', 'dog', 'lazy')]


In [32]:
# Build a trigram model
model = defaultdict(lambda: defaultdict(lambda: 0))


In [33]:
# Count frequency of co-occurrence
for w1, w2, w3 in tri_grams:
    model[(w1, w2)][w3] += 1


In [36]:
# Transform the counts into probabilities
for w1, w2 in model:
    total_count = float(sum(model[w1, w2].values()))
    for w3 in model[w1, w2]:
        model[w1, w2][w3] /= total_count


In [37]:
# Function to predict the next word
def predict_next_word(w1, w2):
    next_word = model[w1, w2]
    if next_word:
        predicted_word = max(next_word, key=next_word.get)
        # Choose the most likely next word
        return predicted_word
    else:
        return "No prediction available"


In [38]:
import math

def calculate_perplexity(test_text, ngram_probs):
    # 1. Tokenize the text into individual words
    tokens = test_text.split()
    n = len(tokens)

    # Handle edge case for short texts to avoid division by zero
    if n <= 1:
        return 0.0

    log_prob_sum = 0

    # 2. Iterate through bigrams (w1, w2)
    for i in range(len(tokens) - 1):
        w1, w2 = tokens[i], tokens[i + 1]

        # Safely look up bigram probabilities from the ngram_probs dictionary
        if w1 in ngram_probs and w2 in ngram_probs[w1]:
            prob = ngram_probs[w1][w2]
        else:
            prob = 1e-6  # Handle unseen bigrams with a small smoothing value

        log_prob_sum += math.log(prob)

    # 3. Calculate perplexity based on the number of bigrams (n - 1)
    perplexity = math.exp(-log_prob_sum / (n - 1))
    return perplexity


In [39]:

         from nltk import word_tokenize
         words="asian exporters fear damage"
         tokens=word_tokenize(words)

In [40]:
calculate_perplexity(words, model)

999999.9999999995

In [41]:
reuters.words('test/15027')

['FIRSTBANC', 'CORP', 'OF', 'OHIO', '&', 'lt', ';', ...]

In [44]:
word1 = reuters.words('test/15027')
print(word1[:20])

['FIRSTBANC', 'CORP', 'OF', 'OHIO', '&', 'lt', ';', 'FBOH', '>', '1ST', 'QTR', 'NET', 'Shr', '74', 'cts', 'vs', '67', 'cts', 'Net', '8']


In [45]:
 predict_next_word('FIRSTBANC', 'CORP')


'No prediction available'

In [46]:
word1 = reuters.words(file_ids[101])
print(word1[:20])

['SOUTHMARK', '&', 'lt', ';', 'SM', '>', 'ACQUIRES', '28', 'NURSING', 'HOMES', 'Southmark', 'Corp', 'said', 'it', 'acquired', '28', 'long', '-', 'term', 'care']
